# 2.7 Tokenizer experiments

Uses `Tokenizer` from `cs336_basics/bpe_tokenizer.py` (the rough-notebook encoder/decoder) with the trained
TinyStories (10K) and OpenWebText (32K) vocab/merges.

In [1]:
import gzip
import multiprocessing as mp
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
from tqdm import tqdm

# run from the repo root whether the kernel starts there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
os.chdir(ROOT)

from cs336_basics.bpe_tokenizer import Tokenizer
from cs336_basics.pretokenization_example import find_chunk_boundaries

In [2]:
SPL = ["<|endoftext|>"]

ts_tok = Tokenizer.from_files("data/bpe_tinystories/vocab.pkl", "data/bpe_tinystories/merges.pkl", SPL)
owt_tok = Tokenizer.from_files("data/bpe_owt/vocab.pkl", "data/bpe_owt/merges.pkl", SPL)

len(ts_tok.vocab), len(ts_tok.merges), len(owt_tok.vocab), len(owt_tok.merges)

(10000, 9743, 32000, 31743)

## Experimental correction: faster merge application

The original `_apply_merges` walks all merges (10K / 32K) for every pre-token, so it costs O(#merges) per
pre-token even when only a handful apply. `FastTokenizer` changes two things, notebook-only:

1. **Rank-based merging** – repeatedly merge the adjacent pair with the lowest merge rank. This gives the same
   result as applying merges in creation order: a token produced by merge *k* can only take part in merges
   created after *k*, so the lowest-rank applicable pair is always the next one the original loop would apply.
2. **Per-pre-token cache** – the same pre-tokens (`" the"`, `"."`, ...) repeat constantly.

In [ ]:
class FastTokenizer(Tokenizer):
    def __init__(self, vocab, merges, special_tokens=None, max_cache=500_000):
        super().__init__(vocab, merges, special_tokens)
        self.ranks = {m: r for r, m in enumerate(self.merges)}
        self.cache = {}
        self.max_cache = max_cache

    def _encode_pretoken(self, pre_tok):
        ids = self.cache.get(pre_tok)
        if ids is None:
            ids = super()._encode_pretoken(pre_tok)
            if len(self.cache) >= self.max_cache:
                self.cache.clear()
            self.cache[pre_tok] = ids
        return ids

    def _apply_merges(self, wrds):
        inf = float("inf")
        while len(wrds) > 1:
            m = min(zip(wrds, wrds[1:]), key=lambda p: self.ranks.get(p, inf))
            if m not in self.ranks:
                break

            new_w = []
            i = 0
            while i < len(wrds):
                if i + 1 < len(wrds) and wrds[i] == m[0] and wrds[i + 1] == m[1]:
                    new_w.append(m[0] + m[1])
                    i += 2
                else:
                    new_w.append(wrds[i])
                    i += 1
            wrds = new_w
        return wrds


ts_fast = FastTokenizer(ts_tok.vocab, ts_tok.merges, SPL)
owt_fast = FastTokenizer(owt_tok.vocab, owt_tok.merges, SPL)

## Sample 10 documents from each dataset

In [4]:
random.seed(0)

with open("data/TinyStoriesV2-GPT4-valid.txt") as f:
    ts_valid_text = f.read()
ts_docs_all = [d for d in ts_valid_text.split(SPL[0]) if d.strip()]
ts_docs = random.sample(ts_docs_all, 10)

# stream ~20 MB from the gzip instead of decompressing the whole file; drop the last (possibly cut) doc
with gzip.open("data/owt_valid.txt.gz", "rt", encoding="utf-8") as f:
    owt_head = f.read(20_000_000)
owt_docs_all = [d for d in owt_head.split(SPL[0])[:-1] if d.strip()]
owt_docs = random.sample(owt_docs_all, 10)

len(ts_docs_all), len(owt_docs_all), [len(d) for d in ts_docs], [len(d) for d in owt_docs]

(27630,
 4043,
 [413, 616, 723, 704, 777, 623, 844, 980, 1225, 728],
 [2015, 3545, 8033, 2913, 3187, 802, 3033, 1875, 1433, 7285])

### Check: `FastTokenizer` == original `Tokenizer`

In [5]:
for d in ts_docs:
    assert ts_fast.encode(d) == ts_tok.encode(d)

# the original loop is slow with 32K merges, so check the two shortest OWT docs
for d in sorted(owt_docs, key=len)[:2]:
    assert owt_fast.encode(d) == owt_tok.encode(d)
    assert owt_tok.decode(owt_tok.encode(d)) == d

print("FastTokenizer matches the original encoder")

FastTokenizer matches the original encoder


## (a) Compression ratio (bytes/token)

In [6]:
def compression_ratio(tok, docs):
    n_bytes = sum(len(d.encode("utf-8")) for d in docs)
    n_toks = sum(len(tok.encode(d)) for d in docs)
    return n_bytes / n_toks, n_bytes, n_toks


ts_on_ts = compression_ratio(ts_fast, ts_docs)
owt_on_owt = compression_ratio(owt_fast, owt_docs)
print(f"TinyStories tokenizer on TinyStories : {ts_on_ts[0]:.2f} bytes/token  ({ts_on_ts[1]} bytes, {ts_on_ts[2]} tokens)")
print(f"OWT tokenizer on OWT                 : {owt_on_owt[0]:.2f} bytes/token  ({owt_on_owt[1]} bytes, {owt_on_owt[2]} tokens)")

TinyStories tokenizer on TinyStories : 4.10 bytes/token  (7633 bytes, 1861 tokens)
OWT tokenizer on OWT                 : 4.41 bytes/token  (34297 bytes, 7782 tokens)


## (b) OWT sample with the TinyStories tokenizer

In [7]:
ts_on_owt = compression_ratio(ts_fast, owt_docs)
owt_on_ts = compression_ratio(owt_fast, ts_docs)
print(f"TinyStories tokenizer on OWT : {ts_on_owt[0]:.2f} bytes/token  (vs {owt_on_owt[0]:.2f} with the OWT tokenizer, "
      f"{ts_on_owt[2] / owt_on_owt[2]:.2f}x more tokens)")
print(f"OWT tokenizer on TinyStories : {owt_on_ts[0]:.2f} bytes/token  (vs {ts_on_ts[0]:.2f} with the TinyStories tokenizer)")

TinyStories tokenizer on OWT : 3.32 bytes/token  (vs 4.41 with the OWT tokenizer, 1.33x more tokens)
OWT tokenizer on TinyStories : 3.96 bytes/token  (vs 4.10 with the TinyStories tokenizer)


In [8]:
# qualitative: how the same OWT snippet gets split by each tokenizer
snippet = owt_docs[0][:300]
for name, tok in [("TinyStories", ts_fast), ("OWT", owt_fast)]:
    pieces = [tok.decode([i]) for i in tok.encode(snippet)]
    print(f"--- {name}: {len(pieces)} tokens")
    print(pieces)

--- TinyStories: 73 tokens
['Every', ' day', ' most', ' of', ' us', ' use', ' a', ' mirror', ' to', ' view', ' our', ' reflection', ' of', ' ph', 'y', 's', 'ic', 'al', ' appearance', ' or', ' st', 'ate', ' of', ' our', ' body', '.', ' When', ' we', ' look', ' in', ' the', ' mirror', ' at', ' the', ' reflection', ' of', ' ourselves', ',', ' we', ' notice', ' any', ' imp', 'er', 'f', 'ect', 'ions', ' and', ' we', ' are', ' quick', ' to', ' remove', ' any', ' food', ' or', ' dirt', ' from', ' our', ' faces', '.', ' We', ' do', ' not', ' stand', ' there', ' and', ' make', ' exc', 'use', ' after', ' exc', 'use', ';']
--- OWT: 63 tokens
['Every', ' day', ' most', ' of', ' us', ' use', ' a', ' mirror', ' to', ' view', ' our', ' reflection', ' of', ' physical', ' appearance', ' or', ' state', ' of', ' our', ' body', '.', ' When', ' we', ' look', ' in', ' the', ' mirror', ' at', ' the', ' reflection', ' of', ' ourselves', ',', ' we', ' notice', ' any', ' imperfect', 'ions', ' and', ' we', ' are

In [9]:
# share of tokens that are a single byte (no merge applied) on the OWT sample
for name, tok in [("TinyStories", ts_fast), ("OWT", owt_fast)]:
    ids = [i for d in owt_docs for i in tok.encode(d)]
    single = sum(len(tok.vocab[i]) == 1 for i in ids) / len(ids)
    print(f"{name:12s} tokenizer: {single:.1%} of OWT tokens are single bytes")

TinyStories  tokenizer: 19.6% of OWT tokens are single bytes
OWT          tokenizer: 14.1% of OWT tokens are single bytes


## (c) Throughput

Single process, cold cache. The original loop is timed on a small slice since it is slow; the fast variant on ~10 MB.

In [10]:
def throughput(tok, text):
    t0 = time.perf_counter()
    tok.encode(text)
    dt = time.perf_counter() - t0
    return len(text.encode("utf-8")) / dt


PILE_BYTES = 825e9
ts_text_10mb = ts_valid_text[:10_000_000]
owt_text_10mb = owt_head[:10_000_000]

rows = [
    ("original  TinyStories", ts_tok, ts_valid_text[:30_000]),
    ("original  OWT", owt_tok, owt_head[:30_000]),
    ("fast      TinyStories", FastTokenizer(ts_tok.vocab, ts_tok.merges, SPL), ts_text_10mb),
    ("fast      OWT", FastTokenizer(owt_tok.vocab, owt_tok.merges, SPL), owt_text_10mb),
]
tput = {}
for name, tok, text in rows:
    bps = throughput(tok, text)
    tput[name] = bps
    print(f"{name:22s}: {bps / 1e6:8.3f} MB/s -> Pile in {PILE_BYTES / bps / 3600:10.1f} h (1 proc), "
          f"{PILE_BYTES / bps / 3600 / 8:9.1f} h (8 procs, ideal)")

original  TinyStories :    0.006 MB/s -> Pile in    39374.6 h (1 proc),    4921.8 h (8 procs, ideal)


original  OWT         :    0.002 MB/s -> Pile in   118232.9 h (1 proc),   14779.1 h (8 procs, ideal)


fast      TinyStories :   13.603 MB/s -> Pile in       16.8 h (1 proc),       2.1 h (8 procs, ideal)


fast      OWT         :    7.766 MB/s -> Pile in       29.5 h (1 proc),       3.7 h (8 procs, ideal)


## (d) Encode train/dev sets to `uint16`

Files are split at `<|endoftext|>` boundaries (so no token straddles a chunk), chunks are encoded in parallel
with `FastTokenizer`, and each chunk's ids are appended in order to a raw `uint16` `.bin` file
(load with `np.memmap(path, dtype=np.uint16, mode="r")`).

In [11]:
_TOK, _IN_PATH = None, None


def _encode_span(span):
    start, end = span
    with open(_IN_PATH, "rb") as f:
        f.seek(start)
        text = f.read(end - start).decode("utf-8", errors="replace")
    # numpy raises OverflowError if an id doesn't fit in uint16
    return np.array(_TOK.encode(text), dtype=np.uint16)


def encode_file_to_bin(tok, in_path, out_path, num_procs=8, chunk_mb=32):
    global _TOK, _IN_PATH
    _TOK, _IN_PATH = tok, in_path  # inherited by forked workers

    size = os.path.getsize(in_path)
    with open(in_path, "rb") as f:
        bounds = find_chunk_boundaries(f, max(num_procs, size // (chunk_mb * 2**20)), SPL[0].encode("utf-8"))
    spans = list(zip(bounds[:-1], bounds[1:]))

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    n_tok, t0 = 0, time.time()
    with open(out_path, "wb") as out, mp.get_context("fork").Pool(num_procs) as pool:
        for arr in tqdm(pool.imap(_encode_span, spans), total=len(spans), desc=os.path.basename(out_path)):
            arr.tofile(out)
            n_tok += len(arr)
    dt = time.time() - t0
    print(f"{in_path} -> {out_path}: {size / 1e6:.1f} MB, {n_tok:,} tokens, {size / n_tok:.2f} bytes/token, "
          f"{dt:.1f}s ({size / dt / 1e6:.2f} MB/s with {num_procs} procs)")
    return n_tok, size, dt


def check_bin(tok, in_path, out_path, n=2000):
    ids = np.memmap(out_path, dtype=np.uint16, mode="r")
    assert ids.max() < len(tok.vocab)
    with open(in_path, encoding="utf-8") as f:
        head = f.read(50_000)
    assert head.startswith(tok.decode(ids[:n].tolist()))
    print(f"{out_path}: {len(ids):,} tokens, max id {ids.max()} < {len(tok.vocab)}, first {n} tokens round-trip")

In [12]:
# OWT valid is only shipped gzipped; the chunked reader needs to seek, so decompress once
def decompress(gz_path):
    txt_path = gz_path[: -len(".gz")]
    if not os.path.exists(txt_path):
        with gzip.open(gz_path, "rb") as src, open(txt_path + ".part", "wb") as dst:
            shutil.copyfileobj(src, dst, length=16 * 2**20)
        os.replace(txt_path + ".part", txt_path)
    return txt_path


owt_valid_path = decompress("data/owt_valid.txt.gz")

In [13]:
par = {}
par["ts_valid"] = encode_file_to_bin(ts_fast, "data/TinyStoriesV2-GPT4-valid.txt", "data/tokens/ts_valid.bin")
check_bin(ts_fast, "data/TinyStoriesV2-GPT4-valid.txt", "data/tokens/ts_valid.bin")

par["owt_valid"] = encode_file_to_bin(owt_fast, owt_valid_path, "data/tokens/owt_valid.bin")
check_bin(owt_fast, owt_valid_path, "data/tokens/owt_valid.bin")


ts_valid.bin:   0%|          | 0/8 [00:00<?, ?it/s]


ts_valid.bin:  12%|█▎        | 1/8 [00:00<00:02,  3.02it/s]


ts_valid.bin: 100%|██████████| 8/8 [00:00<00:00, 23.06it/s]

data/TinyStoriesV2-GPT4-valid.txt -> data/tokens/ts_valid.bin: 22.5 MB, 5,465,883 tokens, 4.12 bytes/token, 0.4s (52.50 MB/s with 8 procs)
data/tokens/ts_valid.bin: 5,465,883 tokens, max id 9999 < 10000, first 2000 tokens round-trip



owt_valid.bin:   0%|          | 0/8 [00:00<?, ?it/s]


owt_valid.bin:  12%|█▎        | 1/8 [00:06<00:47,  6.78s/it]


owt_valid.bin: 100%|██████████| 8/8 [00:06<00:00,  1.18it/s]

data/owt_valid.txt -> data/tokens/owt_valid.bin: 290.0 MB, 66,401,098 tokens, 4.37 bytes/token, 6.9s (42.23 MB/s with 8 procs)
data/tokens/owt_valid.bin: 66,401,098 tokens, max id 31999 < 32000, first 2000 tokens round-trip


In [14]:
# Pile estimate from the end-to-end parallel OWT run (includes I/O and pool overhead)
n_tok, size, dt = par["owt_valid"]
print(f"8-proc OWT throughput {size / dt / 1e6:.2f} MB/s -> Pile in {PILE_BYTES / (size / dt) / 3600:.1f} h")

8-proc OWT throughput 42.23 MB/s -> Pile in 5.4 h


### Full training sets

Gated: TinyStories train is 2.2 GB and OWT train is 12 GB (~5 GB of `uint16` output). Set `RUN_TRAIN = True` to run.

In [15]:
RUN_TRAIN = False

if RUN_TRAIN:
    encode_file_to_bin(ts_fast, "data/TinyStoriesV2-GPT4-train.txt", "data/tokens/ts_train.bin")
    check_bin(ts_fast, "data/TinyStoriesV2-GPT4-train.txt", "data/tokens/ts_train.bin")

    encode_file_to_bin(owt_fast, "data/owt_train.txt", "data/tokens/owt_train.bin")
    check_bin(owt_fast, "data/owt_train.txt", "data/tokens/owt_train.bin")

## Answers

**(a)** On 10 sampled docs, the TinyStories tokenizer (10K) compresses TinyStories to **4.10 bytes/token**, and the OpenWebText tokenizer (32K) compresses OWT to **4.41 bytes/token**. Over the full valid splits the figures are 4.12 and 4.37.

**(b)** Running the TinyStories tokenizer on the OWT sample drops compression to **3.32 bytes/token**, so it needs 1.33x as many tokens. Words that are rare in children's stories break into fragments, e.g. `' physical'` → `' ph','y','s','ic','al'`, and 19.6% of tokens are single bytes, compared with 14.1% for the OWT tokenizer.

**(c)** The original merge-every-pre-token loop runs at about **2–6 KB/s**. At that rate the Pile (825 GB) would take roughly 4.5–13.5 years on one process. With the experimental rank-based merges and pre-token cache, a single process runs at about **7.8 MB/s on OWT (~30 h for the Pile)**. The measured end-to-end rate with 8 processes is **42 MB/s (~5.4 h)**.

**(d)** `uint16` holds IDs from 0 to 65,535, so both vocabularies (10K and 32K IDs) fit without loss. It uses half the space of `int32`: for example, the 66.4M OWT valid tokens take 133 MB.
